# Mollweide: independent reference checks

Compare the site's NumPy solution with PROJ through pyproj, then preserve reference coordinates for the browser checker tests. Inputs and outputs use a unit-radius sphere. The positive antimeridian is normalized to −π, matching the editable code.

In [1]:
from pathlib import Path
import json
import numpy as np
import pyproj

root = Path(
    "/Users/greg/Documents/dev/Map_Projection_Website Development/mobile-redesign"
)
namespace = {}
exec((root / "src/python/mollweide-solution.py").read_text(), namespace)
project = namespace["project"]
reference = pyproj.Proj("+proj=moll +R=1 +lon_0=0")
print(
    "NumPy",
    np.__version__,
    "pyproj",
    pyproj.__version__,
    "PROJ",
    pyproj.proj_version_str,
)
np.testing.assert_allclose(
    project(np.array([0.0]), np.array([0.0])), [[0.0], [0.0]], atol=1e-12
)
print("Origin is correct.")

NumPy 2.5.3 pyproj 3.8.0 PROJ 9.8.1
Origin is correct.


In [2]:
rng = np.random.default_rng(20260911)
lon = rng.uniform(-np.pi, np.pi, 20000)
lat = rng.uniform(-np.pi / 2 + 1e-4, np.pi / 2 - 1e-4, 20000)
x, y = project(lon, lat)
rx, ry = reference(lon, lat, radians=True)
error = np.hypot(x - rx, y - ry)
print("20,000 independent samples; maximum coordinate error:", error.max())
np.testing.assert_allclose(x, rx, atol=2e-7)
np.testing.assert_allclose(y, ry, atol=2e-7)
px, py = project(np.array([-2.0, 0.0, 2.0]), np.array([-np.pi / 2, 0.0, np.pi / 2]))
np.testing.assert_allclose(px, 0, atol=1e-12)
np.testing.assert_allclose(py, [-np.sqrt(2), 0, np.sqrt(2)], atol=1e-12)
print("Pole handling and origin passed.")

20,000 independent samples; maximum coordinate error: 1.4313767318849018e-11
Pole handling and origin passed.


In [3]:
points = [
    (np.radians(lon), np.radians(lat))
    for lat in [-90, -70, -40, 0, 40, 70, 90]
    for lon in [-180, -120, -60, 0, 60, 120, 179]
]
h = 1e-5
for lat in [-0.9, 0, 0.9]:
    for lon in [-1.4, 0.5]:
        points.extend([(lon + h, lat), (lon - h, lat), (lon, lat + h), (lon, lat - h)])
points = np.array(points)
rx, ry = reference(points[:, 0], points[:, 1], radians=True)
fixture = {
    "reference": f"pyproj {pyproj.__version__}; PROJ {pyproj.proj_version_str}; +proj=moll +R=1 +lon_0=0",
    "lon": points[:, 0].tolist(),
    "lat": points[:, 1].tolist(),
    "x": rx.tolist(),
    "y": ry.tolist(),
}
(root / "src/pages/__tests__/mollweide-reference.json").write_text(
    json.dumps(fixture, indent=2) + "\n"
)
print(
    "Saved", len(points), "independent reference samples, including derivative probes."
)

Saved 73 independent reference samples, including derivative probes.
